# Phase 2 - DocLayNet layout detector smoke (T4)

Runner only (CLAUDE.md P1/P2). The Phase 2 **gate**: confirm `Aryn/deformable-detr-DocLayNet`
loads, exposes `config.id2label` with a Table class, detects a Table box, and that the box
crops - on a real T4 so runtime / VRAM are measured. It pins nothing; pinning
`config.LAYOUT_MODEL` is a manual step after the gate passes.

**Why transformers is pinned to 4.49.0:** the checkpoint was saved with transformers 4.36.2
and uses a timm resnet50 backbone. The v5 meta-init loader does not load that backbone's
weights, so detection collapses to degenerate boxes. Pinning below v5 restores the eager
loader that loads the backbone. Logic lives in `scripts/smoke_layout_detector.py`, not here.

In [ ]:
# Boot - clone-if-absent, pin the Phase 2 dev branch
import os
BRANCH = 'feature/phase2-doclaynet-layout'
if not os.path.isdir('/content/FinDocStructRAG/.git'):
    !git clone --quiet https://github.com/AD2000X/FinDocStructRAG.git /content/FinDocStructRAG
!cd /content/FinDocStructRAG && git fetch origin --quiet && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
%cd /content/FinDocStructRAG

In [ ]:
# Install - pin transformers below v5 so the eager loader loads the timm backbone.
# torch is preinstalled on Colab; the smoke subprocess picks up this pinned transformers.
!pip install -q "transformers==4.49.0" timm Pillow requests
!python -c "import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)"

## Step 1 - confirm loading is fixed (example page)

Run the smoke on the model card's example page first. Loading is fixed when: (1) there is **no**
`Materializing param=` line, and (2) the top detections are **large, high-confidence** boxes -
not the top-center ~50px low-score specks that meant the backbone was unloaded. A Table is not
required here (the example page may not have one); this step only checks the model runs correctly.

In [ ]:
!python scripts/smoke_layout_detector.py --threshold 0.3

## Step 2 - the real gate (a page with a Table GT)

Only after Step 1 looks healthy. Pull one DocLayNet val page that has a Table annotation
(`category_id 9 == Table`), then run the smoke on it. The gate passes when a Table box is
detected and cropped (`smoke OK`); the crop is saved so it can be eyeballed.

In [ ]:
# Grab a DocLayNet val page containing a Table (category_id 9 == Table).
!pip install -q datasets
from datasets import load_dataset
from pathlib import Path

out = Path('/content/doclaynet_table_smoke.png')
ds = load_dataset('docling-project/DocLayNet-v1.1', split='val', streaming=True)
for i, ex in enumerate(ds):
    if 9 in ex['category_id']:
        ex['image'].convert('RGB').save(out)
        print('saved', out, '| idx', i, '| size', ex['image'].size, '| cats', set(ex['category_id']))
        break

In [ ]:
!python scripts/smoke_layout_detector.py --image /content/doclaynet_table_smoke.png --threshold 0.3 --save-crop /content/layout_table_crop.png